# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. All entities—record sets, fields, and columns—are referenced by their `@id` as defined in the Croissant schema for consistency.

### Dataset Source
The dataset source is provided via a Croissant schema URL ([https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Keeping as an object

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview

Review available record sets, fields, and their `@id` values. All references will use the `@id` as the unique identifier.

**Note:** You can always inspect `dataset.metadata` for a full list of fields, but the recommended way to interact with the dataset is via `dataset.record_sets` and their contained fields.

In [ ]:
# List all record sets and their fields (referencing by @id)
for record_set in dataset.record_sets:
    print(f"Record Set Name: {record_set.name}")
    print(f"  @id: {record_set['@id']}")
    print(f"  Description: {record_set.description}")
    print(f"  Fields (@id):")
    for field in record_set.fields:
        print(f"    - {field['@id']}: {field.name} ({field.data_type})")
    print("-")

## 3. Data Extraction

For further exploration, select one or more record sets by `@id` and load their data into Pandas DataFrames. This enables standard analysis workflows.

**In this example, we'll load the main protein quantification record set.**  _Update `record_sets_to_load` as appropriate if working with other record sets._

In [ ]:
# Gather all available record set @id values
all_record_set_ids = [rs['@id'] for rs in dataset.record_sets]
# Print them for overview
print("Available record sets (@id):\n", all_record_set_ids)

# For this analysis, we'll use the main protein quantification record set
# For this dataset, often the primary table is under 'cr:main' or similar. Inspect from previous cell if unsure.
# Example placeholder: replace with the actual main table's @id
main_record_set_id = all_record_set_ids[0]  # If only 1, otherwise manually set

dataframes = {}
for record_set_id in all_record_set_ids:
    # Load records for each record set
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

df = dataframes[main_record_set_id]
print(f"Columns in main table ({main_record_set_id}):")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group by meaningful fields, referencing all by their `@id`.

### Example steps:
- Filter by a numeric field (e.g., peptide count, abundance).
- Normalize a numeric field.
- Group by another field (e.g., sample condition or accession).

**Replace the field `@id` variables below with those displayed for your dataset. Here, we'll guess common protein field ids, but validate against section 3 output!**

In [ ]:
# Set your numeric field and grouping field (@id)
# For demonstration, assume the following ids exist (update to actual as needed):
numeric_field_id = None
group_field_id = None

# Attempt to auto-select sensible defaults by name or fallback to user input
for col in df.columns:
    if any(x in col.lower() for x in ['abundance', 'count', 'peptide', 'mw', 'molecular_weight']):
        numeric_field_id = col
        break
for col in df.columns:
    if col != numeric_field_id and any(x in col.lower() for x in ['accession', 'sample', 'condition']):
        group_field_id = col
        break

if numeric_field_id is None:
    print("Could not infer a numeric field from the columns. Please manually set `numeric_field_id` to a suitable column name (use section 3 output)")
else:
    print(f"Using numeric field: {numeric_field_id}")

if group_field_id is not None:
    print(f"Using group field: {group_field_id}")

# Choose an example threshold
threshold = 10
if numeric_field_id is not None:
    numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[numeric_series > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (numeric_series[numeric_series > threshold] - numeric_series.mean()) / numeric_series.std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group if possible
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization

Visualize the distribution of a numeric variable, and if grouping is possible, show group means. This section uses `matplotlib`/`seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        # Boxplot by group field
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

We have demonstrated how to programmatically load and analyze a scientific dataset using the [Croissant](https://mlcommons.org/croissant/) standard and `mlcroissant`. Key fields and record sets were referenced using their `@id`, ensuring clarity for reproducible data science. Adjust the field variables to deep dive into any aspect of the dataset based on your analysis needs.